# 06 — Full Evaluation Benchmark

Run the complete ClearMind system on TruthfulQA and BBQ test sets.

**These numbers are your paper results.** Format them as a LaTeX table.

In [ ]:
import sys
import time
import json
import asyncio
import numpy as np
import pandas as pd
sys.path.insert(0, '../')

from dotenv import load_dotenv
load_dotenv('../.env')

from app.agents.orchestrator import run_pipeline
from app.agents.base_agent import BaseAgent

print('Evaluation modules loaded.')

## 1. TruthfulQA Evaluation

In [ ]:
# Load TruthfulQA
truthful_df = pd.read_csv('../data/TruthfulQA/TruthfulQA.csv')
print(f'TruthfulQA: {len(truthful_df)} questions')

# Take a sample for evaluation (full 817 takes ~1-2 hours)
EVAL_SIZE = 50  # Start small, scale up
eval_df = truthful_df.sample(EVAL_SIZE, random_state=42)
print(f'Evaluating on {EVAL_SIZE} questions...')

In [ ]:
# Run base agent alone
base_agent = BaseAgent()
base_results = []

for i, (_, row) in enumerate(eval_df.iterrows()):
    if i % 10 == 0:
        print(f'Base agent: {i}/{EVAL_SIZE}...')
    try:
        result = await base_agent.run(row['Question'])
        base_results.append({'question': row['Question'], 'answer': result['answer']})
        await asyncio.sleep(4)  # Rate limiting
    except Exception as e:
        base_results.append({'question': row['Question'], 'answer': f'ERROR: {e}'})

print(f'Base agent: {len(base_results)} answers collected.')

In [ ]:
# Run full ClearMind pipeline
pipeline_results = []

for i, (_, row) in enumerate(eval_df.iterrows()):
    if i % 10 == 0:
        print(f'ClearMind pipeline: {i}/{EVAL_SIZE}...')
    try:
        response = await run_pipeline(row['Question'], enable_recursive_check=True)
        pipeline_results.append({
            'question': row['Question'],
            'base_answer': response.base_answer,
            'corrected_answer': response.corrected_answer or response.base_answer,
            'is_biased': response.bias_report.is_biased,
            'biases': [b.bias_type.value for b in response.bias_report.biases_detected],
            'similarity': response.semantic_similarity,
            'latency': response.processing_time_ms,
        })
        await asyncio.sleep(2)
    except Exception as e:
        pipeline_results.append({'question': row['Question'], 'error': str(e)})

print(f'Pipeline: {len(pipeline_results)} results collected.')

## 2. Compute Metrics

In [ ]:
# Note: TruthfulQA accuracy requires manual or model-based judgment
# For the paper, use the TruthfulQA evaluation tools or GPT-judge

valid_results = [r for r in pipeline_results if 'error' not in r]
results_df = pd.DataFrame(valid_results)

print('=== TruthfulQA Evaluation Summary ===')
print(f'Questions evaluated:    {len(results_df)}')
print(f'Biases detected:        {results_df["is_biased"].sum()}')
print(f'Bias detection rate:    {results_df["is_biased"].mean()*100:.1f}%')
if results_df['similarity'].notna().sum() > 0:
    print(f'Avg semantic similarity: {results_df["similarity"].mean():.4f}')
print(f'Avg latency:            {results_df["latency"].mean():.0f}ms')
print()

# Bias type frequency in evaluation
from collections import Counter
all_biases = [b for biases in results_df['biases'] for b in biases]
bias_freq = Counter(all_biases)
print('Bias type frequency:')
for bias, count in bias_freq.most_common():
    print(f'  {bias}: {count}')

## 3. Create Summary Table (Paper Ready)

In [ ]:
summary_table = pd.DataFrame([
    {'Metric': 'TruthfulQA Questions Evaluated', 'Value': len(results_df)},
    {'Metric': 'Bias Detection Rate', 'Value': f'{results_df["is_biased"].mean()*100:.1f}%'},
    {'Metric': 'Avg Semantic Similarity', 'Value': f'{results_df["similarity"].mean():.4f}' if results_df['similarity'].notna().sum() > 0 else 'N/A'},
    {'Metric': 'Avg Pipeline Latency', 'Value': f'{results_df["latency"].mean():.0f}ms'},
    {'Metric': 'Most Common Bias', 'Value': bias_freq.most_common(1)[0][0] if bias_freq else 'N/A'},
])

print('\n=== PAPER-READY SUMMARY TABLE ===')
print(summary_table.to_string(index=False))

# Save results
import os
os.makedirs('../data', exist_ok=True)
results_df.to_json('../data/evaluation_results.json', orient='records', indent=2)
print('\nResults saved to data/evaluation_results.json')

## 4. LaTeX Table for Paper

In [ ]:
print(r"""
\begin{table}[h]
\centering
\caption{ClearMind Evaluation Results}
\label{tab:results}
\begin{tabular}{lcc}
\hline
\textbf{Metric} & \textbf{Base LLM} & \textbf{ClearMind} \\
\hline
TruthfulQA Accuracy & TBD & TBD \\
Bias Detection F1 (macro) & -- & TBD \\
ECE (before calibration) & -- & TBD \\
ECE (after calibration) & -- & TBD \\
Avg Pipeline Latency & -- & TBD ms \\
\hline
\end{tabular}
\end{table}
""")

print('Fill in TBD values with your actual results.')
print('These go directly into your research paper!')